# ShockLink Tecplot 2D cut

This notebook reads the local BATSRUS Tecplot sample, fits the bow shock from velocity divergence, extracts a `shockfit` neighborhood, creates a planar cut from that reduced region, and overlays the fitted shock on pressure contours with PyVista.

From the repository root, install and launch with:

```bash
pip install -e ".[notebook]"
jupyter lab examples/tecplot_2d_cut.ipynb
```

> **Memory note:** `data/3d.dat` is about 1.3 GB. The normalized grid and cut require additional memory.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import time

import numpy as np
import pyvista as pv


def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src/shocklink").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the ShockLink repository or examples directory")


ROOT = find_repository_root()
if importlib.util.find_spec("shocklink") is None:
    sys.path.insert(0, str(ROOT / "src"))

from shocklink.dataset import (
    calc_velocity_divergence,
    get_2d_cut,
    plot_2d_cut,
)
from shocklink.bowshock import fit_bow_shock, extract_shockfit_range
from shocklink.tecplot import read_tecplot

pv.set_jupyter_backend("static")

In [ ]:
# Change these values to explore another file, plane, or color field.
DATA_PATH = ROOT / "data/3d.dat"
NORMAL = "z"
ORIGIN = (0.0, 0.0, 0.0)
SCALARS = "p"
SHOCKFIT_RANGE = [-5.0, 5.0]

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Tecplot sample not found: {DATA_PATH}")

In [ ]:
started = time.perf_counter()
grid = read_tecplot(DATA_PATH)
calc_velocity_divergence(grid)
fit = fit_bow_shock(grid)
load_seconds = time.perf_counter() - started

print(f"Loaded in {load_seconds:.3f} s")
print(f"Bounds: {tuple(float(value) for value in grid.bounds)}")
print("Added point array: div(U)")
print(f"loc0: {fit.loc0}")
print(f"loc1: {fit.loc1}")
print(f"loc2: {fit.loc2}")
print(f"x0: {fit.loc0[0]:.6g}")
print(f"a: {fit.curvature:.6g}")
print(f"Point arrays: {list(grid.point_data.keys())}")
grid

In [ ]:
shock_region = extract_shockfit_range(
    grid,
    lower=SHOCKFIT_RANGE[0],
    upper=SHOCKFIT_RANGE[1],
)

print(f"Shock-region points: {shock_region.n_points:,}")
print(f"Shock-region cells: {shock_region.n_cells:,}")
print(f"Shock-region bounds: {tuple(float(value) for value in shock_region.bounds)}")
print(f"Shock-region point arrays: {list(shock_region.point_data.keys())}")
print(f"Original point IDs: {'vtkOriginalPointIds' in shock_region.point_data}")
print(f"Original cell IDs: {'vtkOriginalCellIds' in shock_region.cell_data}")

cut = get_2d_cut(shock_region, normal=NORMAL, origin=ORIGIN)

print(f"Cut points: {cut.n_points:,}")
print(f"Cut cells: {cut.n_cells:,}")
print(f"Cut bounds: {tuple(float(value) for value in cut.bounds)}")
print(f"Cut point arrays: {list(cut.point_data.keys())}")
cut

In [ ]:
required_arrays = {"P [nPa]", "B [nT]", "U [km/s]", "div(U)", "shockfit"}
assert required_arrays <= set(cut.point_data)

cut_normal = np.asarray(cut.field_data["shocklink_cut_normal"]).reshape(-1)
cut_origin = np.asarray(cut.field_data["shocklink_cut_origin"]).reshape(-1)
distances = (cut.points - cut_origin) @ cut_normal
np.testing.assert_allclose(distances, 0.0, atol=1e-6)

print("Cut validation passed")

In [ ]:
plotter = plot_2d_cut(cut, scalars=SCALARS, show=False, xrange=[-40,20], yrange=[-20,20])
pressure_contours = cut.contour(isosurfaces=15, scalars="P [nPa]")
plotter.add_mesh(pressure_contours, color="black", line_width=1, show_scalar_bar=False)
shock_contour = cut.contour(isosurfaces=[0.0], scalars="shockfit")
plotter.add_mesh(shock_contour, color="white", line_width=5, label="Bow-shock fit")
plotter.show(jupyter_backend="static")